# Étape 8 : Modèle Hybride avec Item Features

## 🎯 Objectif

Améliorer les recommandations en incorporant les **métadonnées des articles** (item features) dans le modèle LightFM. Cette approche hybride combine **collaborative filtering** et **content-based filtering**.

## 📚 Documentation Référence

**LightFM Building Datasets** : `5.Documentations/LightFM/Building datasets — LightFM 1.16 documentation.pdf`

## 🔬 Approche Hybride

### Collaborative Filtering (CF) Pur
- **Basé uniquement** sur les interactions user-item
- ✅ Capture les patterns d'achat similaires
- ❌ **Cold-start problem** : ne peut pas recommander de nouveaux items

### Hybrid Model (CF + Content)
- **Combine** interactions + métadonnées items
- ✅ Peut généraliser aux nouveaux items grâce aux features
- ✅ Améliore la qualité des recommandations
- ✅ Réduit le cold-start problem

## 🎨 Item Features à Utiliser

Métadonnées disponibles dans le dataset H&M :

| Feature | Type | Exemples | Utilité |
|---------|------|----------|---------|
| `product_type_name` | Catégoriel | "T-shirt", "Jeans" | Type de vêtement |
| `product_group_name` | Catégoriel | "Garment Upper body" | Groupe produit |
| `colour_group_name` | Catégoriel | "Black", "White" | Couleur |
| `section_name` | Catégoriel | "Womenwear", "Menswear" | Section |
| `garment_group_name` | Catégoriel | "Jersey Basic" | Groupe vêtement |

## 🧪 Expérimentations

### 1️⃣ **CF Pur vs Hybrid**
Comparer les performances avec et sans features :
- Baseline : Modèle CF pur (Step 6)
- Hybrid : Modèle avec item features

### 2️⃣ **Cold-Start Scenarios**
Tester la capacité à recommander des items avec peu d'interactions :
- Items populaires (beaucoup d'interactions)
- Items de niche (peu d'interactions)
- Items complètement nouveaux (0 interactions train)

### 3️⃣ **Feature Ablation**
Identifier quelles features contribuent le plus :
- Toutes les features
- Sans couleur
- Sans type de produit
- Etc.

## 📊 Métriques

- **Precision@K / Recall@K / AUC** : Performance globale
- **Coverage** : Diversité du catalogue recommandé
- **Cold-start Precision** : Performance sur items avec <10 interactions

## ⚙️ Configuration : Choix de la Taille du Sample

**IMPORTANT** : Utilisez la **même taille** que Steps 5-7.

In [1]:
# ⚙️ CONFIGURATION : Choisir la taille du sample
# Valeurs possibles : '10K', '50K', '100K'
# IMPORTANT: Utiliser la même taille que Steps 5-7

SAMPLE_SIZE = '50K'  # ⭐ Changer à '50K' pour résultats robustes
SPLIT_STRATEGY = 'temporal'  # ou 'random', 'userbased'

print(f"{'='*80}")
print(f"CONFIGURATION")
print(f"{'='*80}")
print(f"\n✅ Taille sélectionnée : {SAMPLE_SIZE}")
print(f"   Stratégie de split : {SPLIT_STRATEGY}")
print(f"   Chemin splits : data/processed/{SAMPLE_SIZE}/splits/{SPLIT_STRATEGY}/")
print(f"   Chemin models : models/{SAMPLE_SIZE}/")

# Vérifier que le modèle CF pur existe (Step 6)
import os
models_path = f'models/{SAMPLE_SIZE}/'
cf_model_path = models_path + 'step6_optimized_model.pkl'

if not os.path.exists(cf_model_path):
    print(f"\n❌ ERREUR: {cf_model_path} n'existe pas!")
    print(f"   Exécutez d'abord Step 6 avec SAMPLE_SIZE = '{SAMPLE_SIZE}'")
    raise FileNotFoundError(f"Modèle CF pur non trouvé pour {SAMPLE_SIZE}")

print(f"\n✅ Configuration validée - Modèle CF pur Step 6 trouvé")

CONFIGURATION

✅ Taille sélectionnée : 50K
   Stratégie de split : temporal
   Chemin splits : data/processed/50K/splits/temporal/
   Chemin models : models/50K/

✅ Configuration validée - Modèle CF pur Step 6 trouvé


## 1. Configuration et Imports

In [2]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import json
import pickle
import os
from collections import defaultdict, Counter

# Imports scipy
from scipy.sparse import load_npz, csr_matrix

# Imports LightFM
try:
    from lightfm import LightFM
    from lightfm.data import Dataset
    from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
    LIGHTFM_AVAILABLE = True
    print("✅ LightFM installé et disponible")
except ImportError:
    LIGHTFM_AVAILABLE = False
    print("⚠️  LightFM n'est pas installé.")
    raise ImportError("LightFM est requis pour ce notebook")

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Seed pour reproductibilité
np.random.seed(42)

print("✅ Configuration terminée")

✅ LightFM installé et disponible
✅ Configuration terminée


/opt/anaconda3/envs/lightfm_env/lib/python3.10/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


## 2. Chargement des Données et Modèle CF Pur

Nous chargeons :
1. Les données (Step 4)
2. Le modèle CF pur optimisé (Step 6) pour comparaison
3. Les métadonnées articles pour créer les features

In [3]:
print("=" * 80)
print(f"CHARGEMENT DONNÉES ET MODÈLE CF PUR ({SAMPLE_SIZE} - {SPLIT_STRATEGY})")
print("=" * 80)

# Chemins
SPLITS_PATH = f'data/processed/{SAMPLE_SIZE}/splits/{SPLIT_STRATEGY}/'
MODELS_PATH = f'models/{SAMPLE_SIZE}/'
PROCESSED_PATH = f'data/processed/{SAMPLE_SIZE}/'
SAMPLED_PATH = f'data/sampled/{SAMPLE_SIZE}/'

# 1. Charger le modèle CF pur (Step 6) pour comparaison
print(f"\n📦 Chargement du modèle CF pur (Step 6)...")
with open(MODELS_PATH + 'step6_optimized_model.pkl', 'rb') as f:
    cf_pure_model = pickle.load(f)
print(f"   ✓ Modèle CF pur chargé")

# Charger config Step 6
with open(MODELS_PATH + 'step6_optimization_results.json', 'r') as f:
    step6_results = json.load(f)
    cf_pure_config = step6_results['best_configuration']

print(f"\n   Configuration CF pur:")
for key, val in cf_pure_config.items():
    if isinstance(val, float) and val < 0.001:
        print(f"      • {key}: {val:.2e}")
    else:
        print(f"      • {key}: {val}")

# 2. Charger les matrices d'interactions (convertir en CSR)
print(f"\n📂 Chargement des matrices d'interactions...")
train_interactions = load_npz(SPLITS_PATH + 'train_interactions.npz').tocsr()
test_interactions = load_npz(SPLITS_PATH + 'test_interactions.npz').tocsr()

print(f"   ✓ Train: {train_interactions.shape} - {train_interactions.nnz:,} interactions")
print(f"   ✓ Test: {test_interactions.shape} - {test_interactions.nnz:,} interactions")

num_users, num_items = train_interactions.shape

# 3. Charger les transactions pour reconstruction
print(f"\n📂 Chargement des transactions...")
transactions = pd.read_csv(PROCESSED_PATH + 'transactions.csv')
print(f"   ✓ {len(transactions):,} transactions chargées")

# 4. Charger les métadonnées articles (avec features)
print(f"\n📂 Chargement des articles avec métadonnées...")
articles = pd.read_csv(SAMPLED_PATH + 'articles_sampled.csv')
print(f"   ✓ {len(articles):,} articles chargés")

# Afficher les colonnes disponibles
print(f"\n   Colonnes disponibles:")
for col in articles.columns[:15]:  # Premiers 15
    print(f"      • {col}")

print(f"\n📊 DATASET ({SAMPLE_SIZE}):")
print(f"   Users: {num_users:,}")
print(f"   Items: {num_items:,}")
print(f"   Train interactions: {train_interactions.nnz:,}")
print(f"   Test interactions: {test_interactions.nnz:,}")

print(f"\n✅ Chargement terminé")

CHARGEMENT DONNÉES ET MODÈLE CF PUR (50K - temporal)

📦 Chargement du modèle CF pur (Step 6)...
   ✓ Modèle CF pur chargé

   Configuration CF pur:
      • no_components: 42.0
      • learning_rate: 0.005012686302434877
      • item_alpha: 4.21e-06
      • user_alpha: 1.44e-06
      • loss: warp
      • epochs: 20.0

📂 Chargement des matrices d'interactions...
   ✓ Train: (46668, 24216) - 44,929 interactions
   ✓ Test: (46668, 24216) - 230 interactions

📂 Chargement des transactions...
   ✓ 50,000 transactions chargées

📂 Chargement des articles avec métadonnées...
   ✓ 24,216 articles chargés

   Colonnes disponibles:
      • article_id
      • product_code
      • prod_name
      • product_type_no
      • product_type_name
      • product_group_name
      • graphical_appearance_no
      • graphical_appearance_name
      • colour_group_code
      • colour_group_name
      • perceived_colour_value_id
      • perceived_colour_value_name
      • perceived_colour_master_id
      • perceiv

## 3. Préparation des Item Features

### 🎨 Sélection des Features

Nous utilisons les métadonnées catégorielles des articles :
- `product_type_name` : Type de produit (T-shirt, Jeans, etc.)
- `colour_group_name` : Groupe de couleur
- `product_group_name` : Groupe de produit
- `section_name` : Section (Womenwear, Menswear, etc.)
- `garment_group_name` : Groupe de vêtement

### 🔧 Préparation

1. Nettoyer les valeurs manquantes
2. Créer des feature strings au format LightFM : `"feature_name:value"`
3. Construire un mapping item_id → liste de features

In [4]:
print("=" * 80)
print("PRÉPARATION DES ITEM FEATURES")
print("=" * 80)

from sklearn.preprocessing import MultiLabelBinarizer
from scipy import sparse
import pickle

# Étape 1 : Charger le mapping de Step 3
print(f"\n🔧 Chargement du mapping Step 3...")
with open(PROCESSED_PATH + 'item_id_mapping.pkl', 'rb') as f:
    item_id_mapping = pickle.load(f)

# Créer reverse mapping
item_id_reverse = {v: k for k, v in item_id_mapping.items()}

print(f"   ✓ Mapping chargé : {len(item_id_mapping):,} items")
print(f"   Exemple : article_id → item_idx")
for article_id, item_idx in list(item_id_mapping.items())[:3]:
    print(f"      {article_id} → {item_idx}")

# Features sélectionnées
FEATURE_COLUMNS = [
    'product_type_name',
    'colour_group_name',
    'product_group_name',
    'section_name',
    'garment_group_name'
]

print(f"\n📋 Features sélectionnées ({len(FEATURE_COLUMNS)}):")
for feat in FEATURE_COLUMNS:
    print(f"   • {feat}")

# Nettoyer articles
print(f"\n🔧 Préparation du DataFrame articles...")
articles_clean = articles.copy()
for col in FEATURE_COLUMNS:
    articles_clean[col] = articles_clean[col].fillna('unknown')

# SOLUTION FINALE : Créer explicitement une liste pour CHAQUE item_idx
print(f"\n🔧 Création de la liste ordonnée (UN élément par item_idx)...")

items_features_list = []
missing_count = 0

for item_idx in range(num_items):
    # Trouver l'article_id correspondant
    article_id = item_id_reverse[item_idx]

    # Chercher dans articles_clean
    article_row = articles_clean[articles_clean['article_id'] == article_id]

    if len(article_row) == 0:
        # Article absent dans articles.csv, utiliser features par défaut
        feature_dict = {col: 'unknown' for col in FEATURE_COLUMNS}
        missing_count += 1
    else:
        # Article trouvé, extraire les features
        feature_dict = article_row.iloc[0][FEATURE_COLUMNS].to_dict()

    items_features_list.append(feature_dict)

print(f"   ✓ Liste créée : {len(items_features_list):,} éléments")
print(f"   ⚠️  Articles manquants dans articles.csv : {missing_count:,}")

# Créer DataFrame ordonné
items_features_df = pd.DataFrame(items_features_list)

print(f"\n   Vérification :")
for idx in range(min(3, len(items_features_df))):
    article_id = item_id_reverse[idx]
    print(f"      Ligne {idx}: item_idx={idx}, article_id={article_id}")

# Statistiques
print(f"\n📊 Statistiques des features:")
for col in FEATURE_COLUMNS:
    n_unique = items_features_df[col].nunique()
    print(f"   • {col:<25} : {n_unique:>4} valeurs uniques")

# Créer feature string combinée
print(f"\n🔧 Création de la feature string combinée...")
items_features_df['features'] = (
    items_features_df['product_type_name'] + '_' +
    items_features_df['colour_group_name'] + '_' +
    items_features_df['product_group_name'] + '_' +
    items_features_df['section_name'] + '_' +
    items_features_df['garment_group_name']
)

print(f"   ✓ Feature strings créées")

# Exemples
print(f"\n📝 Exemples de feature strings:")
for idx in range(min(3, len(items_features_df))):
    article_id = item_id_reverse[idx]
    print(f"   Ligne {idx} (article={article_id}): {items_features_df.iloc[idx]['features']}")

# One-hot encoding
print(f"\n🔧 One-hot encoding avec MultiLabelBinarizer...")
feature_strings = items_features_df['features'].str.split('_')

mlb = MultiLabelBinarizer()
item_features_dense = mlb.fit_transform(feature_strings)
item_features_matrix = sparse.csr_matrix(item_features_dense)

print(f"   ✓ Matrice de features créée")
print(f"      Shape: {item_features_matrix.shape}")
print(f"      NNZ: {item_features_matrix.nnz:,}")
print(f"      Densité: {item_features_matrix.nnz / (item_features_matrix.shape[0] * item_features_matrix.shape[1]) * 100:.2f}%")

# Afficher noms de features
print(f"\n   Exemples de features détectées:")
for feat_name in list(mlb.classes_)[:15]:
    print(f"      • {feat_name}")

print(f"\n✅ Matrice de features prête : {item_features_matrix.shape}")
print(f"   ✓ ALIGNEMENT GARANTI : ligne i = item_idx i = colonne i de train_interactions")
print(f"   ✓ AUCUN trou : une ligne pour CHAQUE item_idx de 0 à {num_items-1}")

# Vérification finale
print(f"\n🔍 VÉRIFICATION FINALE D'ALIGNEMENT:")
print(f"   train_interactions.shape[1] = {num_items}")
print(f"   item_features_matrix.shape[0] = {item_features_matrix.shape[0]}")
assert num_items == item_features_matrix.shape[0], "ERREUR : Dimensions incompatibles !"
print(f"   ✅ ALIGNEMENT PARFAIT ET GARANTI !")

PRÉPARATION DES ITEM FEATURES

🔧 Chargement du mapping Step 3...
   ✓ Mapping chargé : 24,216 items
   Exemple : article_id → item_idx
      681180037 → 0
      666448005 → 1
      700181003 → 2

📋 Features sélectionnées (5):
   • product_type_name
   • colour_group_name
   • product_group_name
   • section_name
   • garment_group_name

🔧 Préparation du DataFrame articles...

🔧 Création de la liste ordonnée (UN élément par item_idx)...
   ✓ Liste créée : 24,216 éléments
   ⚠️  Articles manquants dans articles.csv : 0

   Vérification :
      Ligne 0: item_idx=0, article_id=681180037
      Ligne 1: item_idx=1, article_id=666448005
      Ligne 2: item_idx=2, article_id=700181003

📊 Statistiques des features:
   • product_type_name         :  106 valeurs uniques
   • colour_group_name         :   50 valeurs uniques
   • product_group_name        :   15 valeurs uniques
   • section_name              :   55 valeurs uniques
   • garment_group_name        :   21 valeurs uniques

🔧 Création de

## 4. Vérification des Matrices d'Interactions

### ✅ Approche Simplifiée

Contrairement à l'approche initiale, nous **réutilisons directement** les matrices créées à l'étape 4.

**Pourquoi ?**
- Les matrices Step 4 sont déjà correctes (user_id et item_id alignés)
- Pas besoin de reconstruire via Dataset API pour les interactions
- La matrice de features (sklearn) est compatible directement avec LightFM

**Vérifications effectuées :**
- Dimensions des matrices train/test
- Nombre d'interactions
- Alignement avec la matrice de features

In [5]:
print("=" * 80)
print("VÉRIFICATION DES MATRICES")
print("=" * 80)

# Approche simplifiée : Réutiliser les matrices Step 4 directement
# Pas besoin de Dataset API pour les interactions !

print(f"\n 📊 Matrices d'interactions (Step 4) :")
print(f"   Train : {train_interactions.shape} - {train_interactions.nnz:,} interactions")
print(f"   Test  : {test_interactions.shape} - {test_interactions.nnz:,} interactions")

print(f"\n 📊 Matrice de features (sklearn) :")
print(f"   Shape : {item_features_matrix.shape}")
print(f"   NNZ   : {item_features_matrix.nnz:,}")

# Vérification critique : dimensions compatibles
assert train_interactions.shape[1] == item_features_matrix.shape[0], \
    f"Incompatibilité : train a {train_interactions.shape[1]} items, features a {item_features_matrix.shape[0]}"

print(f"\n ✅ Vérification OK : {train_interactions.shape[1]} items dans train = {item_features_matrix.shape[0]} items dans features")

print(f"\n 💡 DIFFÉRENCE CLÉ avec approche Dataset :")
print(f"   ❌ Dataset API : LightFM apprend des embeddings de features textuelles")
print(f"   ✅ sklearn MLBinarizer : LightFM voit directement les patterns one-hot")
print(f"   → Items avec features similaires ont des 1 dans les mêmes colonnes")
print(f"   → Le modèle peut apprendre des patterns partagés !")

print(f"\n ✅ Matrices prêtes pour l'entraînement hybride")

VÉRIFICATION DES MATRICES

 📊 Matrices d'interactions (Step 4) :
   Train : (46668, 24216) - 44,929 interactions
   Test  : (46668, 24216) - 230 interactions

 📊 Matrice de features (sklearn) :
   Shape : (24216, 239)
   NNZ   : 115,315

 ✅ Vérification OK : 24216 items dans train = 24216 items dans features

 💡 DIFFÉRENCE CLÉ avec approche Dataset :
   ❌ Dataset API : LightFM apprend des embeddings de features textuelles
   ✅ sklearn MLBinarizer : LightFM voit directement les patterns one-hot
   → Items avec features similaires ont des 1 dans les mêmes colonnes
   → Le modèle peut apprendre des patterns partagés !

 ✅ Matrices prêtes pour l'entraînement hybride


## 5. Entraînement du Modèle Hybride

### 🎯 Configuration

Nous utilisons les **mêmes hyperparamètres** que le modèle CF pur (Step 6) pour une comparaison équitable.

La seule différence : ajout de `item_features` lors de l'entraînement.

In [6]:
print("=" * 80)
print("ENTRAÎNEMENT MODÈLE HYBRIDE")
print("=" * 80)

# Utiliser la même config que CF pur pour comparaison équitable
print(f"\n ⚙️  Configuration (même que CF pur - Step 6):")
for key, val in cf_pure_config.items():
    if isinstance(val, float) and val < 0.001:
        print(f"   • {key}: {val:.2e}")
    else:
        print(f"   • {key}: {val}")

# Créer le modèle hybride
hybrid_model = LightFM(
    loss=cf_pure_config['loss'],
    no_components=int(cf_pure_config['no_components']),
    learning_rate=cf_pure_config['learning_rate'],
    item_alpha=cf_pure_config.get('item_alpha', 0.0),
    user_alpha=cf_pure_config.get('user_alpha', 0.0),
    random_state=42
)

print(f"\n🔄 Entraînement en cours...")
print(f"   Avec item_features (sklearn one-hot)")

import time
start_time = time.time()

# DIFFÉRENCE CLÉ : Utiliser train_interactions (Step 4) directement
# PAS train_inter_hybrid (Dataset API)
hybrid_model.fit(
    interactions=train_interactions,  # ← Matrices Step 4 !
    item_features=item_features_matrix,  # ← sklearn MLBinarizer !
    epochs=int(cf_pure_config.get('epochs', 10)),
    num_threads=4,
    verbose=True
)

training_time = time.time() - start_time

print(f"\n ✅ Entraînement terminé en {training_time:.1f}s")

ENTRAÎNEMENT MODÈLE HYBRIDE

 ⚙️  Configuration (même que CF pur - Step 6):
   • no_components: 42.0
   • learning_rate: 0.005012686302434877
   • item_alpha: 4.21e-06
   • user_alpha: 1.44e-06
   • loss: warp
   • epochs: 20.0

🔄 Entraînement en cours...
   Avec item_features (sklearn one-hot)
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19

 ✅ Entraînement terminé en 1.5s


In [7]:
print("="*80)
print("DEBUG CRITIQUE - ALIGNEMENT DES MATRICES")
print("="*80)

# 1. Vérifier les dimensions
print(f"\n1️⃣  DIMENSIONS :")
print(f"   train_interactions : {train_interactions.shape}")
print(f"   test_interactions  : {test_interactions.shape}")
print(f"   item_features_matrix : {item_features_matrix.shape}")

num_users, num_items = train_interactions.shape
print(f"\n   Nombre de users : {num_users:,}")
print(f"   Nombre d'items  : {num_items:,}")
print(f"   Items dans features : {item_features_matrix.shape[0]:,}")

# 2. Vérifier les types
print(f"\n2️⃣  TYPES :")
print(f"   train_interactions type : {type(train_interactions)}")
print(f"   item_features_matrix type : {type(item_features_matrix)}")

# 3. Vérifier le contenu
print(f"\n3️⃣  CONTENU :")
print(f"   train_interactions.nnz : {train_interactions.nnz:,} interactions")
print(f"   item_features_matrix.nnz : {item_features_matrix.nnz:,} entrées")

# 4. VÉRIFICATION CRITIQUE : Compatibilité
print(f"\n4️⃣  COMPATIBILITÉ :")
if num_items == item_features_matrix.shape[0]:
    print(f"   ✅ Dimensions compatibles : {num_items} == {item_features_matrix.shape[0]}")
else:
    print(f"   ❌ PROBLÈME MAJEUR : {num_items} != {item_features_matrix.shape[0]}")
    print(f"   → LightFM NE PEUT PAS utiliser ces features !")

# 5. Test : Entraîner SANS features pour comparer le temps
print(f"\n5️⃣  TEST COMPARATIF (entraînement sans features) :")
import time

model_test = LightFM(no_components=10, loss='warp', random_state=42)

start = time.time()
model_test.fit(train_interactions, epochs=5, verbose=False)
time_without = time.time() - start

print(f"   Temps sans features (5 epochs) : {time_without:.2f}s")
print(f"   Soit {time_without/5:.3f}s par epoch")

start = time.time()
model_test2 = LightFM(no_components=10, loss='warp', random_state=42)
model_test2.fit(train_interactions, item_features=item_features_matrix, epochs=5, verbose=False)
time_with = time.time() - start

print(f"   Temps avec features (5 epochs) : {time_with:.2f}s")
print(f"   Soit {time_with/5:.3f}s par epoch")

if abs(time_with - time_without) < 0.1:
    print(f"\n   ❌ PROBLÈME : Temps identiques !")
    print(f"   → LightFM ignore probablement les features")
else:
    print(f"\n   ✅ Temps différents (features utilisées)")

# 6. Vérifier l'ordre des items dans articles_clean
print(f"\n6️⃣  ORDRE DES ITEMS :")
print(f"   articles_clean.shape : {articles_clean.shape}")
print(f"   articles_clean.index : {articles_clean.index[:5].tolist()}...")

if not articles_clean.index.equals(pd.RangeIndex(len(articles_clean))):
    print(f"   ⚠️  WARNING : Index pas continu !")
    print(f"   → Possible désalignement avec train_interactions")

DEBUG CRITIQUE - ALIGNEMENT DES MATRICES

1️⃣  DIMENSIONS :
   train_interactions : (46668, 24216)
   test_interactions  : (46668, 24216)
   item_features_matrix : (24216, 239)

   Nombre de users : 46,668
   Nombre d'items  : 24,216
   Items dans features : 24,216

2️⃣  TYPES :
   train_interactions type : <class 'scipy.sparse._csr.csr_matrix'>
   item_features_matrix type : <class 'scipy.sparse._csr.csr_matrix'>

3️⃣  CONTENU :
   train_interactions.nnz : 44,929 interactions
   item_features_matrix.nnz : 115,315 entrées

4️⃣  COMPATIBILITÉ :
   ✅ Dimensions compatibles : 24216 == 24216

5️⃣  TEST COMPARATIF (entraînement sans features) :
   Temps sans features (5 epochs) : 0.12s
   Soit 0.023s par epoch
   Temps avec features (5 epochs) : 0.11s
   Soit 0.022s par epoch

   ❌ PROBLÈME : Temps identiques !
   → LightFM ignore probablement les features

6️⃣  ORDRE DES ITEMS :
   articles_clean.shape : (24216, 25)
   articles_clean.index : [0, 1, 2, 3, 4]...


## 6. Comparaison CF Pur vs Hybrid Model

### 🎯 Objectif

Évaluer si l'ajout de features améliore les performances.

### 📊 Métriques

- Precision@K, Recall@K, AUC sur le test set
- Coverage (diversité du catalogue)

In [8]:
print("=" * 80)
print("COMPARAISON CF PUR vs HYBRID MODEL")
print("=" * 80)

K_VALUES = [5, 10, 20]

print(f"\n 🔄 Évaluation des deux modèles (K={K_VALUES})...")

results_comparison = {
    'cf_pure': {},
    'hybrid': {}
}

# Évaluer CF pur (sur les matrices Step 4)
print(f"\n1️⃣  CF PUR (Step 6):")
print(f"{'K':<6} | {'Precision@K':<13} | {'Recall@K':<13} | {'AUC':<10}")
print("-" * 55)

for k in K_VALUES:
    prec = precision_at_k(cf_pure_model, test_interactions, k=k,
                         train_interactions=train_interactions, num_threads=4).mean()
    rec = recall_at_k(cf_pure_model, test_interactions, k=k,
                     train_interactions=train_interactions, num_threads=4).mean()
    auc = auc_score(cf_pure_model, test_interactions,
                   train_interactions=train_interactions, num_threads=4).mean()

    results_comparison['cf_pure'][k] = {
        'precision': prec,
        'recall': rec,
        'auc': auc
    }

    print(f"{k:<6} | {prec:<13.4f} | {rec:<13.4f} | {auc:<10.4f}")

# Évaluer Hybrid (sur les matrices Step 4 + features sklearn)
print(f"\n2️⃣  HYBRID MODEL (avec item features sklearn):")
print(f"{'K':<6} | {'Precision@K':<13} | {'Recall@K':<13} | {'AUC':<10}")
print("-" * 55)

for k in K_VALUES:
    # DIFFÉRENCE CLÉ : Utiliser test_interactions (Step 4) et item_features_matrix (sklearn)
    prec = precision_at_k(hybrid_model, test_interactions, k=k,
                         train_interactions=train_interactions,
                         item_features=item_features_matrix, num_threads=4).mean()
    rec = recall_at_k(hybrid_model, test_interactions, k=k,
                     train_interactions=train_interactions,
                     item_features=item_features_matrix, num_threads=4).mean()
    auc = auc_score(hybrid_model, test_interactions,
                   train_interactions=train_interactions,
                   item_features=item_features_matrix, num_threads=4).mean()

    results_comparison['hybrid'][k] = {
        'precision': prec,
        'recall': rec,
        'auc': auc
    }

    print(f"{k:<6} | {prec:<13.4f} | {rec:<13.4f} | {auc:<10.4f}")

# Calculer l'amélioration
print(f"\n{'='*80}")
print("📊 AMÉLIORATION HYBRID vs CF PUR")
print(f"{'='*80}")

print(f"\n{'K':<6} | {'ΔPrecision@K':<15} | {'ΔRecall@K':<15} | {'ΔAUC':<10}")
print("-" * 60)

for k in K_VALUES:
    delta_prec = results_comparison['hybrid'][k]['precision'] - results_comparison['cf_pure'][k]['precision']
    delta_rec = results_comparison['hybrid'][k]['recall'] - results_comparison['cf_pure'][k]['recall']
    delta_auc = results_comparison['hybrid'][k]['auc'] - results_comparison['cf_pure'][k]['auc']

    print(f"{k:<6} | {delta_prec:>+14.4f} | {delta_rec:>+14.4f} | {delta_auc:>+9.4f}")

print(f"\n✅ Comparaison terminée")

COMPARAISON CF PUR vs HYBRID MODEL

 🔄 Évaluation des deux modèles (K=[5, 10, 20])...

1️⃣  CF PUR (Step 6):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0035        | 0.0174        | 0.6896    
10     | 0.0030        | 0.0304        | 0.6896    
20     | 0.0020        | 0.0391        | 0.6896    

2️⃣  HYBRID MODEL (avec item features sklearn):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0000        | 0.0000        | 0.5623    
10     | 0.0000        | 0.0000        | 0.5623    
20     | 0.0000        | 0.0000        | 0.5623    

📊 AMÉLIORATION HYBRID vs CF PUR

K      | ΔPrecision@K    | ΔRecall@K       | ΔAUC      
------------------------------------------------------------
5      |        -0.0035 |        -0.0174 |   -0.1273
10     |        -0.0030 |        -0.0304 |   -0.1273
20     |        -0.0020 |        -0.0391 |   -0.1273

✅ C

In [ ]:
# Visualiser la comparaison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Préparer les données
cf_prec = [results_comparison['cf_pure'][k]['precision'] for k in K_VALUES]
hybrid_prec = [results_comparison['hybrid'][k]['precision'] for k in K_VALUES]

cf_rec = [results_comparison['cf_pure'][k]['recall'] for k in K_VALUES]
hybrid_rec = [results_comparison['hybrid'][k]['recall'] for k in K_VALUES]

cf_auc = [results_comparison['cf_pure'][k]['auc'] for k in K_VALUES]
hybrid_auc = [results_comparison['hybrid'][k]['auc'] for k in K_VALUES]

x = np.arange(len(K_VALUES))
width = 0.35

# 1. Precision@K
axes[0].bar(x - width/2, cf_prec, width, label='CF Pure', alpha=0.8, color='steelblue', edgecolor='black')
axes[0].bar(x + width/2, hybrid_prec, width, label='Hybrid', alpha=0.8, color='coral', edgecolor='black')
axes[0].set_xlabel('K', fontsize=11)
axes[0].set_ylabel('Precision@K', fontsize=11)
axes[0].set_title('Precision@K Comparison', fontweight='bold', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(K_VALUES)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# 2. Recall@K
axes[1].bar(x - width/2, cf_rec, width, label='CF Pure', alpha=0.8, color='steelblue', edgecolor='black')
axes[1].bar(x + width/2, hybrid_rec, width, label='Hybrid', alpha=0.8, color='coral', edgecolor='black')
axes[1].set_xlabel('K', fontsize=11)
axes[1].set_ylabel('Recall@K', fontsize=11)
axes[1].set_title('Recall@K Comparison', fontweight='bold', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels(K_VALUES)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# 3. AUC
axes[2].bar(x - width/2, cf_auc, width, label='CF Pure', alpha=0.8, color='steelblue', edgecolor='black')
axes[2].bar(x + width/2, hybrid_auc, width, label='Hybrid', alpha=0.8, color='coral', edgecolor='black')
axes[2].set_xlabel('K', fontsize=11)
axes[2].set_ylabel('AUC', fontsize=11)
axes[2].set_title('AUC Comparison', fontweight='bold', fontsize=12)
axes[2].set_xticks(x)
axes[2].set_xticklabels([f'K={k}' for k in K_VALUES])
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n📊 Visualisations générées")

## 7. Analyse Cold-Start

### 🎯 Objectif

Vérifier si le modèle hybride performe mieux sur les **items avec peu d'interactions** (cold-start).

### 📊 Segmentation Items

- **Populaires** : >P75 interactions train
- **Moyens** : P25-P75 interactions
- **Cold-start** : <P25 interactions

In [ ]:
print("=" * 80)
print("ANALYSE COLD-START")
print("=" * 80)

# Convertir les matrices en CSR pour permettre l'indexation
train_interactions_csr = train_interactions.tocsr()
test_interactions_csr = test_interactions.tocsr()
# Utiliser test_interactions de Step 4

# Calculer la popularité des items (nombre d'interactions train)
item_popularity = np.array(train_interactions.sum(axis=0)).flatten()

# Statistiques
print(f"\n📊 Distribution des interactions par item (train):")
print(f"   Min : {item_popularity.min()}")
print(f"   Q25 : {np.percentile(item_popularity, 25):.0f}")
print(f"   Q50 : {np.percentile(item_popularity, 50):.0f}")
print(f"   Q75 : {np.percentile(item_popularity, 75):.0f}")
print(f"   Max : {item_popularity.max()}")

# Définir les seuils
q25 = np.percentile(item_popularity, 25)
q75 = np.percentile(item_popularity, 75)

# Créer les segments d'items
item_segments = {
    'cold_start': np.where(item_popularity < q25)[0],
    'moyens': np.where((item_popularity >= q25) & (item_popularity < q75))[0],
    'populaires': np.where(item_popularity >= q75)[0]
}

print(f"\n📦 Segments d'items créés:")
for seg_name, items in item_segments.items():
    print(f"   • {seg_name.capitalize():<15} : {len(items):>6,} items ({len(items)/num_items*100:>5.1f}%)")

# Pour chaque segment, calculer les métriques
print(f"\n🔄 Évaluation par segment d'items...")

# On doit filtrer les interactions test par segment d'item
K_COLDSTART = 10

print(f"\n{'Segment':<15} | {'N Items':<10} | {'CF Pure P@10':<15} | {'Hybrid P@10':<15} | {'Amélioration':<15}")
print("-" * 90)

coldstart_results = {}

for seg_name, item_indices in item_segments.items():
    if len(item_indices) == 0:
        continue

    # Filtrer test_interactions pour ne garder que les items du segment (utiliser CSR)
    test_segment = test_interactions_csr[:, item_indices]
    test_segment_hybrid = test_interactions_csr[:, item_indices]

    # Vérifier qu'il y a des interactions
    if test_segment.nnz == 0:
        print(f"{seg_name.capitalize():<15} | {len(item_indices):>10,} | {'N/A':<15} | {'N/A':<15} | {'N/A':<15}")
        continue

    # Évaluer CF pur
    try:
        cf_prec = precision_at_k(cf_pure_model, test_segment, k=K_COLDSTART,
                                train_interactions=train_interactions, num_threads=4).mean()
    except:
        cf_prec = 0.0

    # Évaluer Hybrid
    try:
        hybrid_prec = precision_at_k(hybrid_model, test_segment_hybrid, k=K_COLDSTART,
                                    train_interactions=train_interactions,
                                    item_features=item_features_matrix, num_threads=4).mean()
    except:
        hybrid_prec = 0.0

    improvement = hybrid_prec - cf_prec

    coldstart_results[seg_name] = {
        'n_items': len(item_indices),
        'cf_pure_prec': cf_prec,
        'hybrid_prec': hybrid_prec,
        'improvement': improvement
    }

    print(f"{seg_name.capitalize():<15} | {len(item_indices):>10,} | {cf_prec:<15.4f} | {hybrid_prec:<15.4f} | {improvement:>+14.4f}")

print(f"\n✅ Analyse cold-start terminée")

print(f"\n💡 INTERPRÉTATION:")
print(f"   Si Hybrid > CF Pure sur segment 'cold_start', cela indique que")
print(f"   les features aident à généraliser aux items avec peu d'historique.")

In [ ]:
print("=" * 80)
print("FEATURE ABLATION (SIMPLIFIÉ)")
print("=" * 80)

print(f"\n⚠️  Feature ablation est coûteuse en calcul.")
print(f"   Avec {SAMPLE_SIZE}, nous testons seulement 2-3 configurations.")

# Configuration: tester en retirant chaque feature une par une
ablation_configs = [
    ('all', FEATURE_COLUMNS),
    ('without_colour', [f for f in FEATURE_COLUMNS if f != 'colour_group_name']),
    ('without_product_type', [f for f in FEATURE_COLUMNS if f != 'product_type_name'])
]

ablation_results = {}

K_ABLATION = 10

print(f"\n{'Config':<25} | {'Features':<10} | {'Precision@10':<15} | {'vs All':<15}")
print("-" * 80)

for config_name, features_to_use in ablation_configs:
    print(f"\n🔄 Entraînement: {config_name}...")

    # Recréer item_features avec seulement ces features
    item_feat_ablation = {}
    for idx, row in articles_clean.iterrows():
        features = []
        for col in features_to_use:
            value = str(row[col]).strip().lower().replace(' ', '_')
            features.append(f"{col}:{value}")
        item_feat_ablation[idx] = features

    # Recréer dataset
    all_feat_ablation = set()
    for fl in item_feat_ablation.values():
        all_feat_ablation.update(fl)

    dataset_abl = Dataset()
    dataset_abl.fit(users=user_ids, items=item_ids, item_features=all_feat_ablation)

    (train_abl, _) = dataset_abl.build_interactions(train_triplets)
    (test_abl, _) = dataset_abl.build_interactions(test_triplets)

    item_feat_tuples_abl = [(iid, feats) for iid, feats in item_feat_ablation.items()]
    item_feat_matrix_abl = dataset_abl.build_item_features(item_feat_tuples_abl)

    # Entraîner modèle
    model_abl = LightFM(
        loss=cf_pure_config['loss'],
        no_components=int(cf_pure_config['no_components']),
        learning_rate=cf_pure_config['learning_rate'],
        random_state=42
    )

    model_abl.fit(
        interactions=train_abl,
        item_features=item_feat_matrix_abl,
        epochs=int(cf_pure_config.get('epochs', 10)),
        num_threads=4,
        verbose=False
    )

    # Évaluer
    prec = precision_at_k(model_abl, test_abl, k=K_ABLATION,
                         train_interactions=train_abl,
                         item_features=item_feat_matrix_abl, num_threads=4).mean()

    ablation_results[config_name] = {
        'n_features': len(features_to_use),
        'precision': prec
    }

    # Comparer à 'all'
    if config_name == 'all':
        vs_all = 0.0
    else:
        vs_all = prec - ablation_results['all']['precision']

    print(f"{config_name:<25} | {len(features_to_use):<10} | {prec:<15.4f} | {vs_all:>+14.4f}")

print(f"\n✅ Feature ablation terminée")

print(f"\n💡 INTERPRÉTATION:")
print(f"   Si une config 'without_X' a une baisse significative,")
print(f"   cela indique que la feature X est importante.")

## 9. Synthèse et Recommandations

In [ ]:
print("=" * 80)
print("SYNTHÈSE FINALE - STEP 8")
print("=" * 80)

print(f"\n📊 RÉSUMÉ ({SAMPLE_SIZE}):")

# 1. Comparaison CF pur vs Hybrid
print(f"\n1️⃣  CF PUR vs HYBRID (K=10):")
cf_p10 = results_comparison['cf_pure'][10]['precision']
hybrid_p10 = results_comparison['hybrid'][10]['precision']
improvement = hybrid_p10 - cf_p10

print(f"   • CF Pure Precision@10   : {cf_p10:.4f}")
print(f"   • Hybrid Precision@10    : {hybrid_p10:.4f}")
print(f"   • Amélioration           : {improvement:+.4f} ({improvement/cf_p10*100:+.1f}%)")

# 2. Cold-start
if coldstart_results:
    print(f"\n2️⃣  COLD-START ANALYSIS:")
    if 'cold_start' in coldstart_results:
        cs = coldstart_results['cold_start']
        print(f"   Items cold-start:")
        print(f"      • CF Pure    : {cs['cf_pure_prec']:.4f}")
        print(f"      • Hybrid     : {cs['hybrid_prec']:.4f}")
        print(f"      • Amélioration: {cs['improvement']:+.4f}")

# 3. Feature ablation
if ablation_results:
    print(f"\n3️⃣  FEATURE ABLATION:")
    for config_name, res in ablation_results.items():
        print(f"   • {config_name:<25} : P@10={res['precision']:.4f}")

print(f"\n💡 CONCLUSIONS:")

if improvement > 0:
    print(f"   ✅ Le modèle hybride AMÉLIORE les performances")
    print(f"      → Les item features apportent de l'information utile")
elif improvement > -0.01:
    print(f"   ⚠️  Le modèle hybride a des performances SIMILAIRES au CF pur")
    print(f"      → Les features n'apportent pas beaucoup")
else:
    print(f"   ❌ Le modèle hybride DÉGRADE les performances")
    print(f"      → Possible overfitting ou features bruitées")

if SAMPLE_SIZE == '10K':
    print(f"\n⚠️  RAPPEL: Test set très petit avec 10K")
    print(f"   → Utilisez 50K ou 100K pour résultats fiables")

print(f"\n🎯 RECOMMANDATIONS:")
print(f"   1. Si amélioration significative: utiliser Hybrid en production")
print(f"   2. Si cold-start amélioré: Hybrid utile pour nouveaux items")
print(f"   3. Feature engineering: tester d'autres features (prix, marque, etc.)")
print(f"   4. Pour production: ré-entraîner avec 100K")

print(f"\n✅ Synthèse terminée")

## 10. Sauvegarde des Résultats

In [ ]:
print("=" * 80)
print("SAUVEGARDE DES RÉSULTATS")
print("=" * 80)

print(f"\n💾 Sauvegarde dans {MODELS_PATH}...")

# 1. Sauvegarder le modèle hybride
print(f"\n💾 Sauvegarde du modèle hybride...")
with open(MODELS_PATH + 'step8_hybrid_model.pkl', 'wb') as f:
    pickle.dump(hybrid_model, f)
print(f"   ✓ step8_hybrid_model.pkl")

# 2. Sauvegarder la matrice item_features
print(f"\n💾 Sauvegarde de la matrice item_features...")
from scipy.sparse import save_npz
save_npz(MODELS_PATH + 'step8_item_features.npz', item_features_matrix)
print(f"   ✓ step8_item_features.npz")

# 3. Sauvegarder le dataset LightFM
print(f"\n💾 Sauvegarde du dataset LightFM...")
with open(MODELS_PATH + 'step8_dataset.pkl', 'wb') as f:
    pickle.dump(dataset, f)
print(f"   ✓ step8_dataset.pkl")

# 4. Sauvegarder les résultats
print(f"\n💾 Sauvegarde des résultats...")

results_summary = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'sample_size': SAMPLE_SIZE,
    'split_strategy': SPLIT_STRATEGY,
    'features_used': FEATURE_COLUMNS,
    'n_features': len(all_features),
    'cf_pure_vs_hybrid': {
        str(k): {
            'cf_pure': {
                'precision': float(results_comparison['cf_pure'][k]['precision']),
                'recall': float(results_comparison['cf_pure'][k]['recall']),
                'auc': float(results_comparison['cf_pure'][k]['auc'])
            },
            'hybrid': {
                'precision': float(results_comparison['hybrid'][k]['precision']),
                'recall': float(results_comparison['hybrid'][k]['recall']),
                'auc': float(results_comparison['hybrid'][k]['auc'])
            },
            'improvement': {
                'precision': float(results_comparison['hybrid'][k]['precision'] - results_comparison['cf_pure'][k]['precision']),
                'recall': float(results_comparison['hybrid'][k]['recall'] - results_comparison['cf_pure'][k]['recall']),
                'auc': float(results_comparison['hybrid'][k]['auc'] - results_comparison['cf_pure'][k]['auc'])
            }
        }
        for k in K_VALUES
    },
    'coldstart_analysis': {
        seg: {
            'n_items': data['n_items'],
            'cf_pure_precision': float(data['cf_pure_prec']),
            'hybrid_precision': float(data['hybrid_prec']),
            'improvement': float(data['improvement'])
        }
        for seg, data in coldstart_results.items()
    } if coldstart_results else {},
    'ablation': {
        config: {
            'n_features': data['n_features'],
            'precision': float(data['precision'])
        }
        for config, data in ablation_results.items()
    } if ablation_results else {}
}

with open(MODELS_PATH + 'step8_hybrid_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f"   ✓ step8_hybrid_results.json")

print(f"\n" + "=" * 80)
print("✅ STEP 8 TERMINÉ AVEC SUCCÈS")
print(f"=" * 80)

print(f"\n📦 Fichiers créés dans {MODELS_PATH}:")
print(f"   • step8_hybrid_model.pkl (modèle hybride)")
print(f"   • step8_item_features.npz (matrice features)")
print(f"   • step8_dataset.pkl (dataset LightFM)")
print(f"   • step8_hybrid_results.json (résultats comparatifs)")

print(f"\n🎯 Modèle Hybride ({SAMPLE_SIZE}):")
print(f"   • {len(FEATURE_COLUMNS)} types de features utilisées")
print(f"   • {len(all_features):,} features uniques")
improvement_p10 = results_comparison['hybrid'][10]['precision'] - results_comparison['cf_pure'][10]['precision']
print(f"   • Amélioration Precision@10: {improvement_p10:+.4f}")

print(f"\n🚀 Pipeline complet terminé!")
print(f"   Steps 1-8 : Système de recommandation complet")